In [ ]:
# Imports
import os
from pathlib import Path
markers = (".git", "Program")
current_dir = Path.cwd()
project_root = next((path for path in (current_dir, *current_dir.parents) if any((path / marker).exists() for marker in markers)), current_dir)
os.chdir(project_root)

from backtest import *
import concurrent.futures
import datetime as dt
from functools import partial
from fundamentals import *
from helper_functions import modify_current_date, get_df, get_excel_filename, get_infix, get_volume5m_data, generate_end_dates, merge_stocks, stock_market
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import multiprocessing
import numpy as np
import pandas as pd
from pandas import ExcelWriter as EW
from plot import *
import random
from scipy.optimize import minimize
from scipy.stats import false_discovery_control, kendalltau, linregress, pearsonr, spearmanr, ttest_ind, wilcoxon
from stock_screener import check_conds_tech, check_conds_fund, EM_rating, get_stock_info, stoploss_target
from technicals import *
from tqdm import tqdm

# Connect to TradingView
from tvDatafeed import TvDatafeed, Interval
tv = TvDatafeed()

# Start of the program
start = dt.datetime.now()

# Index
index_name = "^GSPC"
index_dict = {"^HSI": "HKEX", "^GSPC": "S&P 500", "^IXIC": "NASDAQ Composite"}

# Modify the current date
current_date = modify_current_date(start, index_name)

In [ ]:
# Select the stock excel file
excel_filename = get_excel_filename("2026-01-10", "^GSPC", index_dict, 252, 90, True)

In [ ]:
# Read the stocks
excel_df = pd.read_excel(excel_filename)
stocks_df = excel_df.loc[excel_df["Market Cap (B, USD)"] > 10, ["Stock", "Long-term RS", "Market Cap (B, USD)"]]
stocks = stocks_df["Stock"].tolist()

In [ ]:
# === Parameters ===
days_per_week = 5
num_weeks = 52
period_week_zscore = days_per_week * num_weeks  # Lookback period for weekly return z-score calculation
period_pca = 126  # Lookback period for correlation-based clustering
period_mom_short = 21  # Short-term momentum lookback (1 month)
period_mom_long = 252  # Long-term momentum lookback (1 year)
period_vol = 60  # Volatility calculation period (3 months)
num_clusters = 5

# === Fetch Price Data ===
price_data = {}
for stock in tqdm(stocks, desc="Fetching price data"):
    df = get_df(stock, current_date)
    price_data[stock] = df["Close"]

# === Weekly Returns Z-Score Analysis ===
# Measures how unusual this week's return is compared to historical weekly returns
df_prices_weekly = pd.DataFrame({stock: price_data[stock].tail(period_week_zscore + days_per_week) for stock in stocks})
weekly_prices = df_prices_weekly.iloc[::days_per_week]  # Sample every 5th day
weekly_returns = weekly_prices.pct_change().dropna()

mean_return = weekly_returns.mean()
std_return = weekly_returns.std()
recent_return = weekly_returns.iloc[-1]
z_scores = (recent_return - mean_return) / std_return  # Positive = outperforming, Negative = underperforming

# === Momentum and Volatility Analysis ===
# Momentum: ratio of recent price to year-ago price (trend strength)
# Vol-adj Momentum: momentum normalized by volatility (risk-adjusted trend)
momentum_list = []
volatility_list = []

for stock in tqdm(stocks, desc="Calculating momentum and volatility"):
    close = price_data[stock]
    momentum = close.iloc[-period_mom_short] / close.iloc[-period_mom_long] if len(close) >= period_mom_long else np.nan
    volatility = close.pct_change().tail(period_vol).std()
    momentum_list.append(momentum)
    volatility_list.append(volatility)

# === Hierarchical Clustering Analysis ===
# Groups stocks by correlation similarity using Ward's linkage method
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform

df_prices_cluster = pd.DataFrame({stock: price_data[stock].tail(period_pca) for stock in stocks})
returns = df_prices_cluster.pct_change().dropna()
corr_matrix = returns.corr()
dist_matrix = np.sqrt(2 * (1 - corr_matrix))  # Convert correlation to distance
condensed_dist = squareform(dist_matrix, checks=False)
Z = linkage(condensed_dist, method="ward")
cluster_labels = fcluster(Z, t=num_clusters, criterion="maxclust")

# === Combine All Metrics into Single DataFrame ===
df_combined = pd.DataFrame({
    "Stock": stocks,
    "Cluster": cluster_labels,
    "Momentum": momentum_list,
    "Volatility": volatility_list,
    "Mean Weekly Return (%)": (mean_return * 100).values,
    "Std Weekly Return (%)": (std_return * 100).values,
    "This Week Return (%)": (recent_return * 100).values,
    "Z-Score": z_scores.values
})
df_combined["Vol-adj Momentum"] = df_combined["Momentum"] / df_combined["Volatility"]
df_combined = df_combined.sort_values("Vol-adj Momentum", ascending=False).reset_index(drop=True)
df_combined = df_combined.set_index(df_combined.index + 1)
df_combined.index.name = "Rank"

# === Display Combined DataFrame ===
pd.set_option("display.max_rows", None)
display(df_combined)

# === Cluster Summary ===
print("\nCluster Summary:")
for cluster_id in sorted(df_combined["Cluster"].unique()):
    cluster_stocks = df_combined[df_combined["Cluster"] == cluster_id]["Stock"].tolist()
    print(f"Cluster {cluster_id} ({len(cluster_stocks)} stocks): {', '.join(cluster_stocks)}")

# === Dendrogram Visualization ===
plt.figure(figsize=(14, 8))
dendrogram(Z, labels=stocks, leaf_rotation=90, leaf_font_size=8)
plt.title(f"Hierarchical Clustering of {len(stocks)} Stocks (Past {period_pca} Days)")
plt.ylabel("Distance")
plt.tight_layout()
plt.show()

# === Filter and Display Stocks by Cluster ===
# Filter out stocks with unusually high weekly returns (z-score > 2)
# These may be experiencing temporary spikes rather than sustainable momentum
df_filtered = df_combined[df_combined["Z-Score"] <= 2]

print("\nStocks by Cluster (Ranked by Vol-adj Momentum, Z-Score <= 2):\n")
for cluster_id in sorted(df_filtered["Cluster"].unique()):
    cluster_df = df_filtered[df_filtered["Cluster"] == cluster_id].sort_values(
        "Vol-adj Momentum", ascending=False
    )
    cluster_stocks = cluster_df["Stock"].tolist()
    print(f"Cluster {cluster_id} ({len(cluster_df)} stocks): {', '.join(cluster_stocks)}")

In [ ]:
# === Select Top Stock from Each Cluster ===
# Get the highest rank stock from each cluster (already sorted by Vol-adj Momentum)
top_stocks_per_cluster = df_combined.groupby("Cluster").first().reset_index()

# === Calculate Inverse Volatility Weights ===
# Lower volatility stocks receive higher weights (risk parity approach)
top_stocks = top_stocks_per_cluster["Stock"].tolist()
volatilities = top_stocks_per_cluster["Volatility"].tolist()

inv_vol = [1 / v for v in volatilities]
total_inv_vol = sum(inv_vol)
weights = [iv / total_inv_vol for iv in inv_vol]

# === Display Portfolio Allocation ===
result_df = pd.DataFrame({
    "Cluster": top_stocks_per_cluster["Cluster"],
    "Stock": top_stocks,
    "Volatility": volatilities,
    "Weight (%)": [w * 100 for w in weights]
})
display(result_df)